# Lesson 1: AdventureWorks sales star
Neeraj Bhuma

Exercises in course order. This is a fresh implementation using Microsoft's source data and schema script. The team's existing notebooks are unchanged.


In [ ]:
# Exercise 1: set up raw data. No paid resources are created.
import urllib.request, re, json
from pyspark.sql import functions as F
try:
    spark.sql('CREATE CATALOG IF NOT EXISTS adventure_works')
    catalog, lesson_schema = 'adventure_works', 'default'
except Exception as exc:
    print('Catalog creation is unavailable in this workspace:', str(exc)[:400])
    catalog, lesson_schema = 'workspace', 'adventure_works'
lesson = f'{catalog}.{lesson_schema}'
bronze = f'{catalog}.bronze' if catalog == 'adventure_works' else 'workspace.adventure_works_bronze'
silver = f'{catalog}.silver' if catalog == 'adventure_works' else 'workspace.adventure_works_silver'
gold = f'{catalog}.gold' if catalog == 'adventure_works' else 'workspace.adventure_works_gold'
for schema in [lesson, bronze, silver, gold]:
    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {schema}')
spark.sql(f'CREATE VOLUME IF NOT EXISTS {lesson}.raw_data')
raw_dir = f'/Volumes/{catalog}/{lesson_schema}/raw_data'
base = 'https://raw.githubusercontent.com/microsoft/sql-server-samples/master/samples/databases/adventure-works/oltp-install-script'
sources = {'SalesOrderDetail':'Sales', 'Product':'Production', 'SalesTerritory':'Sales', 'SalesOrderHeader':'Sales', 'PurchaseOrderDetail':'Purchasing', 'PurchaseOrderHeader':'Purchasing', 'Vendor':'Purchasing'}
for name in list(sources) + ['instawdb']:
    suffix = '.sql' if name == 'instawdb' else '.csv'
    urllib.request.urlretrieve(base+'/'+name+suffix, raw_dir+'/'+name+suffix)
script = open(raw_dir+'/instawdb.sql').read()
# Read column names from CREATE TABLE, including computed columns in the CSV.
columns = {}
for name, schema in sources.items():
    start = script.index(f'CREATE TABLE [{schema}].[{name}]')
    block = script[start:script.index(') ON [PRIMARY]', start)]
    columns[name] = re.findall(r'^\s*\[([^\]]+)\]', block, re.M)
print('Lesson tables:', lesson, 'Bronze:', bronze, 'Silver:', silver, 'Gold:', gold)
display(dbutils.fs.ls(raw_dir))


In [ ]:
# Exercise 2: apply the source names and types, then save the sales star.
# The CREATE TABLE script is kept alongside the CSVs for checking the mapping.
def raw(name):
    return spark.read.option('sep','\t').option('header',False).csv(raw_dir+'/'+name+'.csv')
def typed(name):
    df = raw(name)
    assert len(df.columns) == len(columns[name]), (name,len(df.columns),columns[name])
    df = df.toDF(*columns[name])
    exprs = []
    for c in df.columns:
        typ = 'string'
        if c.endswith('ID') or c in ['OrderQty','RevisionNumber','Status','CreditRating','SafetyStockLevel','ReorderPoint','DaysToManufacture']:
            typ = 'int'
        elif c.endswith('Date'):
            typ = 'timestamp'
        elif c in ['LineTotal','UnitPrice','UnitPriceDiscount','SubTotal','TaxAmt','Freight','TotalDue','StandardCost','ListPrice','SalesYTD','SalesLastYear','CostYTD','CostLastYear']:
            typ = 'decimal(38,6)' if c == 'LineTotal' and name == 'SalesOrderDetail' else 'decimal(19,4)'
        elif c in ['ReceivedQty','RejectedQty','StockedQty','Weight']:
            typ = 'decimal(18,4)'
        elif c.endswith('Flag') or c == 'PreferredVendorStatus':
            typ = 'boolean'
        exprs.append(F.col(c).cast(typ).alias(c))
    return df.select(*exprs)
products, territories = typed('Product'), typed('SalesTerritory')
sales_detail, sales_header = typed('SalesOrderDetail'), typed('SalesOrderHeader')
assert sales_header.count() == sales_header.select('SalesOrderID').distinct().count()
fact_sales = sales_detail.join(sales_header.select('SalesOrderID','OrderDate','TerritoryID'), 'SalesOrderID')
assert fact_sales.count() == sales_detail.count()
for name, df in [('dim_product',products),('dim_sales_territory',territories),('fact_sales',fact_sales)]:
    df.write.mode('overwrite').option('overwriteSchema',True).saveAsTable(lesson+'.'+name)
    print(name, df.count())
spark.sql(f'USE CATALOG {catalog}')
spark.sql(f'USE SCHEMA {lesson_schema}')


In [ ]:
# Exercise 3: workspace notes and table inspection.
print('Workspace: stores folders, notebooks and Git folders.')
print('Catalog: lists schemas, tables, volumes and permissions.')
print('Compute: runs code; this notebook uses the Free Edition serverless option.')
print('Jobs & Pipelines: schedules notebooks and coordinates dependent tasks.')
print('Data Ingestion: brings source data into the lakehouse.')
display(spark.sql(f'SHOW TABLES IN {lesson}'))
for name in ['fact_sales','dim_product','dim_sales_territory']:
    display(spark.sql(f'DESCRIBE {lesson}.{name}'))
    display(spark.sql(f'SELECT COUNT(*) AS row_count FROM {lesson}.{name}'))


In [ ]:
# Exercise 4: grain, dimensions and joins.
display(spark.sql('SELECT COUNT(*) AS total_rows, COUNT(DISTINCT SalesOrderDetailID) AS distinct_detail_ids FROM fact_sales'))
for dim,key in [('dim_product','ProductID'),('dim_sales_territory','TerritoryID')]:
    display(spark.sql(f'''SELECT '{dim}' AS dimension,
      (SELECT COUNT(*) FROM {dim}) AS dimension_rows,
      (SELECT COUNT(*) FROM fact_sales f LEFT JOIN {dim} d ON f.{key}=d.{key}) AS left_join_rows,
      (SELECT COUNT(*) FROM fact_sales f RIGHT JOIN {dim} d ON f.{key}=d.{key}) AS right_join_rows,
      (SELECT COUNT(*) FROM fact_sales f LEFT JOIN {dim} d ON f.{key}=d.{key} WHERE d.{key} IS NULL) AS orphan_fact_rows'''))
    display(spark.sql(f'SELECT {key}, COUNT(*) AS copies FROM {dim} GROUP BY {key} HAVING COUNT(*)>1'))
print('A LEFT JOIN preserves unmatched facts too. An unchanged count rules out fan-out, but orphan checks are needed to prove matches.')
print('A RIGHT JOIN keeps unused dimension rows. Those rows have NULL fact columns, so its count can be higher.')
print('A duplicate ProductID makes every matching sale appear twice. SUM(LineTotal) then counts the same sale twice without a SQL error.')


In [ ]:
%sql
SELECT * FROM fact_sales ORDER BY ModifiedDate DESC, SalesOrderDetailID DESC LIMIT 10

In [ ]:
# Exercise 5: first queries. The preceding SQL cell gives the ten latest sales.
display(spark.sql('SELECT COUNT(*) AS sales_rows, COUNT(DISTINCT ProductID) AS products_sold, (SELECT COUNT(DISTINCT ProductID) FROM dim_product) AS catalog_products FROM fact_sales'))
revenue_sql = '''SELECT f.ProductID, p.Name, SUM(f.LineTotal) AS Revenue
FROM fact_sales f JOIN dim_product p ON f.ProductID=p.ProductID
GROUP BY f.ProductID,p.Name ORDER BY Revenue DESC'''
display(spark.sql(revenue_sql+' LIMIT 10'))
display(spark.sql('SELECT COUNT(*) AS bad_rows FROM fact_sales WHERE LineTotal IS NULL OR LineTotal<0'))
print('If bad amounts appear, quarantine and check the source before calculating revenue. Do not replace missing amounts with zero without a business rule.')
top5 = spark.sql(revenue_sql+' LIMIT 5').toPandas()
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10,4))
ax.barh(top5['Name'][::-1], top5['Revenue'].astype(float)[::-1], color='#287b8e')
ax.set_xlabel('Net sales revenue (source monetary units)')
ax.set_title('Top 5 products by revenue')
fig.tight_layout()
display(fig)
plt.close(fig)


In [ ]:
# Exercise 6: profile the sales star.
for name,key in [('fact_sales','SalesOrderDetailID'),('dim_product','ProductID'),('dim_sales_territory','TerritoryID')]:
    display(spark.sql(f'DESCRIBE {name}'))
    display(spark.sql(f"SELECT '{name}' AS table_name, COUNT(*) AS rows, COUNT(DISTINCT {key}) AS distinct_keys FROM {name}"))
display(spark.sql('SELECT COUNT(*) AS orphaned_sales FROM fact_sales f LEFT JOIN dim_product p ON f.ProductID=p.ProductID WHERE p.ProductID IS NULL'))
display(spark.sql('SELECT COUNT(*) AS unsold_products FROM dim_product p LEFT JOIN fact_sales f ON p.ProductID=f.ProductID WHERE f.ProductID IS NULL'))
print('An orphan is a sale whose product is missing. An unsold product exists in the catalog but has no sales lines; that is not an orphan sale.')
display(spark.sql('''SELECT f.ProductID,p.Name,COUNT(*) AS order_line_count,COUNT(DISTINCT f.SalesOrderID) AS distinct_orders,SUM(f.OrderQty) AS units FROM fact_sales f JOIN dim_product p ON f.ProductID=p.ProductID GROUP BY f.ProductID,p.Name ORDER BY order_line_count DESC LIMIT 10'''))
display(spark.sql('''SELECT COALESCE(p.ProductLine,'Unspecified') AS ProductLine,SUM(f.LineTotal) AS revenue FROM fact_sales f JOIN dim_product p ON f.ProductID=p.ProductID GROUP BY p.ProductLine ORDER BY revenue DESC'''))


In [ ]:
# Second Exercise 6: the confirmed team.
print('Hello World')
print('Lakshmi Narayana Reddy Mandi')
print('Neeraj Bhuma')
print('Bala Sai Kiran')
# Git folder connection and instructor sharing are separate steps, not implied by this output.


Hello World
Lakshmi Narayana Reddy Mandi
Neeraj Bhuma
Bala Sai Kiran
